# 03 LightGBM vs. MLP at equal budget
Goal: pick the teacher and the student family fairly. This is not a verdict on trees vs. nets. Both models get the same budget from `configs/config.yaml`: `tune.n_trials` trials and `tune.time_cap_seconds` seconds, whichever runs out first.

In [1]:
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.lib import evaluate_regression, load_data, load_split
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg["seed"])

X, y = load_data()
X, y = X.to_numpy(), y.to_numpy()
_test_idx, folds = load_split()  # same 5 folds as every other model; the test rows are not used here
print("budget:", cfg["tune"])

budget: {'n_trials': 20, 'time_cap_seconds': 900}


## LightGBM search
Random search. Each trial samples seven settings, then is scored by the same 5-fold CV. Rounds use early stopping on a 10% slice of each fold's *training* rows, so the validation fold never influences training.

In [2]:
def sample_lgbm_params(rng):
    """Draw one random setting. Rates and penalties are sampled on a log scale."""
    return {
        "learning_rate": float(np.exp(rng.uniform(np.log(0.01), np.log(0.2)))),
        "num_leaves": int(rng.integers(8, 129)),
        "n_estimators": int(np.exp(rng.uniform(np.log(200), np.log(2000)))),  # max rounds
        "min_child_samples": int(rng.integers(5, 101)),
        "subsample": float(rng.uniform(0.5, 1.0)),  # row subsampling
        "colsample_bytree": float(rng.uniform(0.5, 1.0)),  # column subsampling
        "reg_lambda": float(np.exp(rng.uniform(np.log(1e-3), np.log(10)))),  # L2 penalty
    }


def cv_rmse_lgbm(params):
    """5-fold CV RMSE for one setting: (mean, std) over the folds."""
    scores = []
    for train_idx, val_idx in folds:
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=cfg["seed"])
        model = lgb.LGBMRegressor(
            **params, subsample_freq=1, random_state=cfg["seed"], verbose=-1
        )
        model.fit(
            X[fit_idx],
            y[fit_idx],
            eval_X=X[stop_idx],
            eval_y=y[stop_idx],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        pred = model.predict(X[val_idx])  # uses the best round found by early stopping
        scores.append(evaluate_regression(y[val_idx], pred)["rmse"])
    return float(np.mean(scores)), float(np.std(scores, ddof=1))


def run_lgbm_search(n_trials, time_cap_seconds, seed):
    """Random search. Stops after n_trials, or after the trial that crosses the time cap."""
    rng = np.random.default_rng(seed)
    rows = []
    start = time.perf_counter()
    for trial in range(1, n_trials + 1):
        t0 = time.perf_counter()
        params = sample_lgbm_params(rng)
        cv_mean, cv_std = cv_rmse_lgbm(params)
        now = time.perf_counter()
        rows.append(
            {
                "trial": trial,
                **params,
                "cv_rmse_mean": cv_mean,
                "cv_rmse_std": cv_std,
                "trial_seconds": now - t0,
                "cumulative_seconds": now - start,
            }
        )
        if now - start >= time_cap_seconds:
            break
    return pd.DataFrame(rows)

### Tiny run: 3 trials
Checks that the loop works and the log has the right columns. It is not saved. The real search moves to `src/tune.py`.

In [3]:
tiny = run_lgbm_search(3, cfg["tune"]["time_cap_seconds"], cfg["seed"])
tiny.round(4)

,trial,learning_rate,num_leaves,n_estimators,min_child_samples,subsample,colsample_bytree,reg_lambda,cv_rmse_mean,cv_rmse_std,trial_seconds,cumulative_seconds
0,1,0.1016,87,1444,47,0.8487,0.5471,7.9889,0.4663,0.0117,28.3058,28.3058
1,2,0.0978,94,268,80,0.7252,0.6854,5.0940,0.4549,0.0120,20.6305,48.9363
2,3,0.0688,56,555,83,0.6136,0.7773,0.0018,0.4606,0.0132,16.5662,65.5025
